In [ ]:
import neuralforecast as nf
from neuralforecast import NeuralForecast
import logging
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)
from neuralforecast.losses.pytorch import MAE
from neuralforecast.auto import AutoGRU, AutoTCN, AutoFEDformer, AutoPatchTST
import pandas as pd
import numpy as np
from sktime.performance_metrics.forecasting import (
    mean_absolute_scaled_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
)

import matplotlib.pyplot as plt
from sktime.utils.plotting import plot_series

In [55]:
from typing import Optional


def wide_to_long_df(df: pd.DataFrame) -> pd.DataFrame:
    kursy = df[['USD_EUR']]
    kursy = kursy.rename(columns={"USD_EUR": "exo", "DATE": "ds"})
    df = df.drop(columns=['USD_EUR'])
    df = pd.melt(df, ignore_index=False).reset_index(names="date")
    df = df.rename(columns={"variable": "unique_id", "date": "ds", "value": "y"})
    df = df.join(kursy, on='ds')
    return df


def long_to_wide_df(df: pd.DataFrame, values_col: Optional[str] = None) -> pd.DataFrame:
    if "unique_id" not in df.columns:
        df = df.reset_index(names="unique_id")

    values_col = values_col if values_col else df.columns[-1]
    df = pd.pivot(df, columns="unique_id", index="ds", values=values_col)
    return df

In [46]:
df_stocks = pd.read_csv('portfolio_data.csv')
df_btc = df_stocks[['Date','BTC']]
df_btc.index = pd.to_datetime(df_btc['Date'])
df_btc = df_btc.drop(columns=['Date'])

df_kurs = pd.read_csv('USD-EUR.csv')
df_kurs = df_kurs[['Date','Price']]
df_kurs = df_kurs.sort_index(ascending=False)
df_kurs.index = pd.to_datetime(df_kurs['Date'])
df_kurs = df_kurs.drop(columns=['Date'])

df = df_btc.join(df_kurs, how='inner')
df.rename(columns={'BTC':'BTC_Price','Price':'USD_EUR'}, inplace=True)

In [47]:
df_train = df.loc[df.index < '2019-01-01']
df_test = df.loc[df.index >= '2019-01-01']

In [68]:
def evaluate_model(
    model,
    df_train: pd.Series,
    df_test: pd.Series,
    h=20
) -> None:
    

    data_train = wide_to_long_df(df_train)
    data_train['y'].loc[data_train['y'].isna()==True]=0

    data_test = wide_to_long_df(df_test)
    data_test['y'].loc[data_test['y'].isna()==True]=0
    

    model.fit(data_train)
    
    forecasts = model.predict()

    mae = mean_absolute_error(df_test[:h], long_to_wide_df(forecasts))
    mase = mean_absolute_scaled_error(df_test[:h], long_to_wide_df(forecasts),y_train=df_train)

    print(f"MAE: {mae:.2f}")
    print(f"MASE: {mase:.2f}")
    
    return long_to_wide_df(forecasts)

In [ ]:
h=14

def config():
    return {
        "hist_exog_list": ['exo']  # Historic exogenous
    }

model = AutoGRU(
    h=h,
    loss=MAE(),
    num_samples=10,
    verbose=False,
    config=config()
)

nf = NeuralForecast(
    models=[model],
    freq="D"
    
)

nf.fit

print("GRU forecaster 14 days horizon:")

gru_forecast = evaluate_model(nf,df_train=df_train,df_test=df_test,h=h)
